In [2]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# ==============================
# 2. LOAD DATASET
# ==============================
df = pd.read_csv("tickets.csv")

print("Dataset Preview:")
print(df.head())


# ==============================
# 3. TEXT CLEANING
# ==============================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_ticket'] = df['ticket'].apply(clean_text)


# ==============================
# 4. TEXT VECTORIZATION
# ==============================
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_ticket'])


# ==============================
# 5. CATEGORY CLASSIFICATION
# ==============================
y_category = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42, stratify=y_category
)

model_cat = LogisticRegression(max_iter=200)
model_cat.fit(X_train, y_train)

pred_cat = model_cat.predict(X_test)

print("\n===== CATEGORY MODEL =====")
print("Accuracy:", accuracy_score(y_test, pred_cat))
print(classification_report(y_test, pred_cat, zero_division=0))


# ==============================
# 6. PRIORITY CLASSIFICATION
# ==============================
y_priority = df['priority']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42, stratify=y_priority
)

model_pri = LogisticRegression(max_iter=200)
model_pri.fit(X_train, y_train)

pred_pri = model_pri.predict(X_test)

print("\n===== PRIORITY MODEL =====")
print("Accuracy:", accuracy_score(y_test, pred_pri))
print(classification_report(y_test, pred_pri, zero_division=0))


# ==============================
# 7. TEST NEW TICKET
# ==============================
new_ticket = ["My account is locked and payment failed"]

new_clean = [clean_text(t) for t in new_ticket]
new_vector = vectorizer.transform(new_clean)

print("\n===== NEW TICKET PREDICTION =====")
print("Ticket:", new_ticket[0])
print("Predicted Category:", model_cat.predict(new_vector)[0])
print("Predicted Priority:", model_pri.predict(new_vector)[0])

Dataset Preview:
                                   ticket   category priority
0       Payment failed but money deducted    Billing     High
1           Unable to login to my account    Account   Medium
2    Website is very slow and not loading  Technical     High
3                Need refund for my order    Billing     High
4  Account locked after multiple attempts    Account   Medium

===== CATEGORY MODEL =====
Accuracy: 0.75
              precision    recall  f1-score   support

     Account       0.50      1.00      0.67         1
     Billing       1.00      1.00      1.00         1
   Technical       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.83      0.78         4
weighted avg       0.88      0.75      0.75         4


===== PRIORITY MODEL =====
Accuracy: 0.75
              precision    recall  f1-score   support

        High       1.00      0.50      0.67         2
      Medium       0.67      1.00  